# Stage 9b, observe Spark: what the Spark History MCP server can tell you

Stage 9 asked an agent about the pipeline. Here the focus is Spark itself. The notebook runs a small workload on the
lab's Spark Connect server, reads its metrics with the **Spark History MCP** tools **by hand**, then lets an agent (built
on [AWS Strands](https://strandsagents.com), `lab_agents.py`) diagnose it.

The one MCP endpoint serves two backends: `server="live"` is the running Spark Connect session (the queries you just
ran), `server="history"` is the History Server (finished applications). On S3 a running application's event log only
reaches the History Server when the application stops, so the History Server cannot list the current session: that is
why the live backend exists. The server is **read-only**.

The workload runs three queries: a full scan, the same scan with a partition filter, and a join with one very hot key.
Without an LLM (`llm.provider=none`) everything up to the agent runs; the agent cell prints `skipped (no LLM configured)`.

In [ ]:
import sys, pathlib, json
import pandas as pd
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import lab_data
import lab_utils as lu

lu.require_ops()
if "spark_history" not in lu.ops_servers():
    raise RuntimeError("The Spark History MCP server is off: set sparkHistoryMcp.enabled=true (and sparkHistory.enabled).")
llm_ready = lu.llm_available()
if llm_ready:
    import lab_agents as la
    lu.require_tool_model()
print("tools:", ", ".join(sorted(t["name"] for t in lu.ops_tools("spark_history"))))

def short(obj, n=1200):
    text = obj if isinstance(obj, str) else json.dumps(obj, indent=1, default=str)
    print(text if len(text) <= n else text[:n] + f"\n... ({len(text) - n} more characters)")

## 1. A workload with something to find
`lab_data.order_events()` replicated 200 times into an Iceberg table partitioned by `status`; a scan over all of it, the
same scan restricted to one partition, and a sort-merge join where 90% of the rows share the key `0`. For the join,
broadcast, adaptive skew handling and partition coalescing are off and the shuffle has 8 partitions, so the hot key
lands in one task instead of being hidden in a single merged one. Names are fixed, so the notebook can run again.

The live backend is **one** application, the lab's Spark Connect server, holding every session of every notebook and
of the Airflow DAG. The cell keeps this session's id (`spark.session_id`) to find its own queries there.


In [ ]:
from pyspark.sql import functions as F

spark = lu.get_spark("ops02-workload")
session_id = spark.session_id
table = f"{lu.CATALOG}.shop_bronze.ops02_events"
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {lu.CATALOG}.shop_bronze")
spark.sql(f"DROP TABLE IF EXISTS {table}")
rows = [{k: e[k] for k in ("order_id", "customer_id", "product_id", "quantity", "order_ts", "status")} for e in lab_data.order_events()]
schema = "order_id long, customer_id long, product_id long, quantity int, order_ts string, status string"
(spark.createDataFrame(rows, schema).crossJoin(spark.range(200).withColumnRenamed("id", "copy"))
      .writeTo(table).partitionedBy(F.col("status")).using("iceberg").create())

for key, value in {"spark.sql.autoBroadcastJoinThreshold": "-1",             # a shuffle join, not a broadcast
                   "spark.sql.adaptive.skewJoin.enabled": "false",           # keep the skew instead of splitting it
                   "spark.sql.adaptive.coalescePartitions.enabled": "false", # 8 tasks, not one merged task
                   "spark.sql.shuffle.partitions": "8"}.items():
    spark.conf.set(key, value)
full = spark.table(table).agg(F.count("*").alias("rows"), F.sum("quantity").alias("units")).collect()[0]
filtered = spark.table(table).where("status = 'returned'").agg(F.count("*").alias("rows"), F.sum("quantity").alias("units")).collect()[0]
left = spark.range(2_000_000).select(F.when(F.col("id") % 10 < 9, F.lit(0)).otherwise(F.col("id")).alias("k"))
skewed = left.join(spark.range(1000).withColumnRenamed("id", "k"), "k").count()
print("full scan:", full.asDict(), "\nfiltered :", filtered.asDict(), "\nskewed join rows:", skewed)
print("Spark Connect session:", session_id)


## 2. The tools by hand, on the live session
Every MCP tool is a function with a JSON schema. First the applications the **live** backend knows (one: the Spark
Connect server), then this session's SQL executions: `list_sql_executions` filtered on the session id, in order. The
last three are the full scan, the filtered scan and the join. The ids change when Spark Connect restarts, so the
notebook never hardcodes one.


In [ ]:
def call(name, **args):
    return lu.ops_call("spark_history", name, {"server": "live", **args})

def items(res):
    """The list inside a tool result (some tools return a bare list, others wrap it)."""
    if isinstance(res, list):
        return res
    if isinstance(res, dict):
        for v in res.values():
            if isinstance(v, list):
                return v
    return []

apps = items(call("list_applications"))
short(apps)
app_id = (apps[0].get("id") or apps[0].get("app_id")) if apps and isinstance(apps[0], dict) else None
print("live application:", app_id)

In [ ]:
mine = sorted((q for q in items(call("list_sql_executions", app_id=app_id, description=session_id, limit=100))
               if isinstance(q, dict) and session_id in str(q.get("description", ""))), key=lambda q: q["id"]) if app_id else []
if len(mine) < 3:
    raise RuntimeError("this session's queries are not on the live backend; re-run the workload cell")
full_id, filtered_id, join_id = (q["id"] for q in mine[-3:])
for name, q in zip(("full scan", "filtered scan", "skewed join"), mine[-3:]):
    print(f"{name:14} SQL execution {q['id']}: {q.get('duration')} ms")


## 3. Where did the time go?
`get_sql_execution(include_stages=True)` lists the join's stages; `get_stage(with_summaries=True)` gives each stage's
task metrics as quantiles (min, quartiles, max). A skewed stage has one task that reads far more shuffle records than
the median task. (`get_job_bottlenecks` ranks the slowest stages of the whole application, which here means every
session of the lab, so it is the wrong tool for one query.) `compare_sql_executions` then puts the two scans side by
side: the partition filter should read a fraction of the bytes.


In [ ]:
def quantile(dist, values, q):
    """The value at quantile q (0.5 = median, 1.0 = max) of a get_stage task-metric distribution."""
    qs = dist["quantiles"]
    return values[min(range(len(qs)), key=lambda i: abs(qs[i] - q))]

stats = []
for s in call("get_sql_execution", app_id=app_id, execution_id=join_id, include_stages=True).get("stages", []):
    st = call("get_stage", app_id=app_id, stage_id=s["stageId"], with_summaries=True)
    if not isinstance(st, dict) or "error" in st or (st.get("numTasks") or 0) < 2:
        continue   # skipped stages (reused shuffle output) and single-task stages have nothing to compare
    d = st.get("taskMetricsDistributions") or {}
    records = (d.get("shuffleReadMetrics") or {}).get("readRecords") or []
    if records and quantile(d, records, 0.5):
        stats.append({"stage": s["stageId"], "tasks": st["numTasks"],
                      "median records": quantile(d, records, 0.5), "max records": quantile(d, records, 1.0),
                      "median ms": quantile(d, d["duration"], 0.5), "max ms": quantile(d, d["duration"], 1.0)})
skew = pd.DataFrame(stats)
if skew.empty:
    skewed_stage = None
    print("no multi-task shuffle stage in the join: was the workload cell run with its settings?")
else:
    skew["max / median"] = (skew["max records"] / skew["median records"]).round(1)
    skew = skew.sort_values("max / median", ascending=False)
    display(skew)
    skewed_stage = int(skew.iloc[0]["stage"])
    print(f"stage {skewed_stage}: its slowest task reads {skew.iloc[0]['max / median']}x the median task's records (the hot key 0)")


In [ ]:
cmp = call("compare_sql_executions", app_id1=app_id, app_id2=app_id, execution_id1=full_id, execution_id2=filtered_id)
if "error" in cmp:
    raise RuntimeError(cmp["error"])
cols = ("sqlId", "duration", "tasks", "inputBytes", "stageTime")
display(pd.DataFrame([{k: cmp[side].get(k) for k in cols} for side in ("a", "b")], index=["full scan", "filtered scan"]))
print(f"the partition filter read {cmp['b']['inputBytes'] / cmp['a']['inputBytes']:.0%} of the full scan's bytes")


## 4. Finished applications, on the History Server
Completed applications (the earlier notebooks' Spark Connect sessions, the SparkApplications of the e2e) live on the
`history` backend. A fresh lab, or one older than the 7-day cleaner, has none: that is fine.

In [ ]:
try:
    done = items(lu.ops_call("spark_history", "list_applications", {"server": "history"}))
except Exception as e:
    done = []
    print("history backend not reachable:", type(e).__name__, e)
print(f"{len(done)} finished application(s) in the History Server" + ("" if done else
      " (a running application's events reach S3 only when it stops, so the current session is not listed)"))
short(done[:3], 1000)

## 5. Ask the agent
`lab_agents.ops_agent(("spark_history",), spark_server="live")` offers the read-only Spark tools only. Its prompt
says which server holds what (`live` for the running session, `history` for finished applications); a small model
still leaves `server` out, and the MCP server then asks `history`, so `spark_server="live"` fills it in (the steps
table notes it). The questions name the application and
the SQL execution ids from section 2: the live application holds every session of the lab, so "my last query" has no
answer there. Every tool call is printed under the answer, so you can check that the answer comes from the tools.


In [ ]:
QUESTIONS = [
    f"In the live Spark application {app_id}, SQL execution {join_id} is a join. Which of its stages has the most "
    "task skew? Compare its slowest task with the median task.",
    f"In the live Spark application {app_id}, SQL executions {full_id} and {filtered_id} are a full scan and the same "
    "scan with a partition filter. How many input bytes did each read?",
    f"In the live Spark application {app_id}, why did SQL execution {join_id} take longer than SQL execution {full_id}?",
]
if lu.skip_unless_llm("the agent diagnoses the workload"):
    with la.ops_agent(("spark_history",), spark_server="live") as agent:
        for q in QUESTIONS:
            out = la.ask(agent, q)
            print(f"Q: {q}\n\nA: {out['answer']}\n")
            display(pd.DataFrame([{"tool": s["tool"], "args": json.dumps(s["args"]), "note": s.get("note", ""),
                                   "result": json.dumps(s.get("result"), default=str)[:200]} for s in out["steps"]]))


## What to look for
* Section 3 names the skewed stage: one of its 8 tasks reads tens of times the median task's shuffle records. The
  agent's first answer should name the same stage; the numbers are in the tool results, not in the model's imagination.
* The filtered scan reads one partition of the table: about a third of the full scan's input bytes.
* The server is **read-only**: the agent can look, not kill a job or change a setting.
* Small models sometimes pick the wrong backend (`history` for the live session) or skip a step; the steps table is how
  you check. A bigger tool model (`llm.toolModel`) does this more reliably.
